# 18 differential abundance

**Purpose:** Run sample-level composition and differential-abundance analyses.

**Original analysis notebook:** `18_differential_abundance_composition_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 37 — Differential abundance / composition analysis by condition

This notebook performs **sample-level and patient-level composition analysis** using the final annotation/CNV consensus object.

Main goal: test whether the proportions of final cellular compartments, refined annotations, and key clusters differ across disease conditions (`GBM`, `LGG`, `ndGBM`, `rGBM`).

Important principle: differential abundance is evaluated mainly at the **sample/patient level**, not only at the single-cell level, to reduce pseudoreplication caused by treating thousands of cells from the same sample as independent observations.


## What this notebook does

1. Loads the final annotated object:
   `${GLIOMA_PROJECT_ROOT}/processed/integrated/GBM_core_4datasets_final_annotation.h5ad`
2. Uses only `adata.obs`, so it does not modify expression matrices.
3. Aggregates cell counts per sample and per patient.
4. Calculates proportions of:
   - `final_compartment_major`
   - `final_annotation_refined`
   - `leiden_scvi_main` clusters
   - `copykat_pred_clean` calls
5. Performs condition-level statistical testing:
   - Kruskal-Wallis global test across conditions
   - pairwise Mann-Whitney U tests between conditions
   - Benjamini-Hochberg FDR correction
   - optional binomial GLM as a secondary sensitivity analysis
6. Generates thesis-ready tables and figures.

Interpretation rule: this notebook tests **composition differences**, not gene expression differences. It answers questions like: *Is the malignant-like compartment proportion different between GBM, LGG, ndGBM, and rGBM at sample/patient level?*


In [ ]:
# =========================
# Imports and settings
# =========================
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

import matplotlib.pyplot as plt

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures" / "DA_composition_HVG8000_r0_6"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation.h5ad"
RUN_TAG = "HVG8000_r0_6"

print("INPUT_H5AD:", INPUT_H5AD)
print("exists:", INPUT_H5AD.exists())


In [ ]:
# =========================
# Helper functions
# =========================

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="	", index=False)
    print(f"[SAVED] {path} shape={df.shape}")


def savefig_replace(path, dpi=180):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print(f"[FIGURE] {path}")


def choose_col(obs, candidates, required=True):
    for c in candidates:
        if c in obs.columns:
            return c
    if required:
        raise KeyError(f"None of candidate columns found: {candidates}")
    return None


def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    out = np.full_like(pvals, np.nan, dtype=float)
    mask = np.isfinite(pvals)
    if mask.sum() > 0:
        out[mask] = multipletests(pvals[mask], method="fdr_bh")[1]
    return out


def safe_kruskal(groups):
    valid = [np.asarray(g, dtype=float) for g in groups if len(g) >= 2 and np.nanstd(g) > 0]
    if len(valid) < 2:
        return np.nan, np.nan, "insufficient_nonconstant_groups"
    try:
        stat, p = stats.kruskal(*valid)
        return float(stat), float(p), "ok"
    except Exception as e:
        return np.nan, np.nan, f"error:{repr(e)}"


def safe_mannwhitney(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2 or len(y) < 2:
        return np.nan, np.nan, "insufficient_n"
    if np.nanstd(x) == 0 and np.nanstd(y) == 0 and np.nanmean(x) == np.nanmean(y):
        return np.nan, 1.0, "both_constant_equal"
    try:
        stat, p = stats.mannwhitneyu(x, y, alternative="two-sided")
        return float(stat), float(p), "ok"
    except Exception as e:
        return np.nan, np.nan, f"error:{repr(e)}"


def cramers_v_from_table(table):
    table = np.asarray(table, dtype=float)
    if table.sum() == 0:
        return np.nan
    chi2, p, dof, expected = stats.chi2_contingency(table)
    n = table.sum()
    r, k = table.shape
    denom = n * max(min(k - 1, r - 1), 1)
    return float(np.sqrt(chi2 / denom))


In [ ]:
# =========================
# Load final object and obs only
# =========================
if not INPUT_H5AD.exists():
    raise FileNotFoundError(INPUT_H5AD)

adata = sc.read_h5ad(INPUT_H5AD, backed="r")
obs = adata.obs.copy()

print(adata)
print("obs shape:", obs.shape)
print("obs columns:", len(obs.columns))

condition_col = choose_col(obs, ["condition", "disease_condition", "group"])
sample_col = choose_col(obs, ["sample_id", "sample", "orig.ident", "sample_name"])
patient_col = choose_col(obs, ["patient_id", "patient", "donor_id"], required=False)
dataset_col = choose_col(obs, ["core_dataset", "dataset_id", "dataset", "study_id"], required=False)
cluster_col = choose_col(obs, ["leiden_scvi_main", "leiden_scvi_HVG8000_r0_6", "leiden"])
compartment_col = choose_col(obs, ["final_compartment_major", "final_malignancy_group_copykat_marker"])
annotation_col = choose_col(obs, ["final_annotation_refined", "annotation_marker_based", "annotation"])
copykat_col = choose_col(obs, ["copykat_pred_clean", "copykat.pred", "copykat_pred"], required=False)

print("condition_col:", condition_col)
print("sample_col:", sample_col)
print("patient_col:", patient_col)
print("dataset_col:", dataset_col)
print("cluster_col:", cluster_col)
print("compartment_col:", compartment_col)
print("annotation_col:", annotation_col)
print("copykat_col:", copykat_col)


In [ ]:
# =========================
# Readiness report
# =========================
readiness_rows = []
for name, value in [
    ("n_cells", obs.shape[0]),
    ("n_genes", adata.n_vars),
    ("condition_col", condition_col),
    ("sample_col", sample_col),
    ("patient_col", patient_col if patient_col else "NA"),
    ("dataset_col", dataset_col if dataset_col else "NA"),
    ("cluster_col", cluster_col),
    ("compartment_col", compartment_col),
    ("annotation_col", annotation_col),
    ("copykat_col", copykat_col if copykat_col else "NA"),
    ("n_conditions", obs[condition_col].astype(str).nunique()),
    ("n_samples", obs[sample_col].astype(str).nunique()),
    ("n_patients", obs[patient_col].astype(str).nunique() if patient_col else np.nan),
    ("n_clusters", obs[cluster_col].astype(str).nunique()),
    ("n_compartments", obs[compartment_col].astype(str).nunique()),
    ("n_refined_annotations", obs[annotation_col].astype(str).nunique()),
]:
    readiness_rows.append({"metric": name, "value": value})

readiness = pd.DataFrame(readiness_rows)
READINESS_TSV = METADATA_DIR / f"DA_readiness_{RUN_TAG}.tsv"
write_tsv_replace(readiness, READINESS_TSV)
display(readiness)


In [ ]:
# =========================
# Core aggregation functions
# =========================

def build_unit_feature_table(obs, unit_col, feature_col, condition_col, dataset_col=None, patient_col=None, min_unit_cells=50):
    # Return long table with count/proportion of feature categories per biological unit.
    keep_cols = [unit_col, feature_col, condition_col]
    if dataset_col and dataset_col not in keep_cols:
        keep_cols.append(dataset_col)
    if patient_col and patient_col not in keep_cols and patient_col != unit_col:
        keep_cols.append(patient_col)

    df = obs[keep_cols].copy()
    for c in keep_cols:
        df[c] = df[c].astype(str)

    unit_meta = df.groupby(unit_col).agg(
        n_cells_total=(feature_col, "size"),
        condition=(condition_col, lambda x: x.value_counts().idxmax()),
    ).reset_index()

    if dataset_col:
        unit_meta["dataset"] = df.groupby(unit_col)[dataset_col].agg(lambda x: x.value_counts().idxmax()).values
    else:
        unit_meta["dataset"] = "NA"

    if patient_col and patient_col != unit_col:
        unit_meta["patient"] = df.groupby(unit_col)[patient_col].agg(lambda x: x.value_counts().idxmax()).values
    elif unit_col == patient_col:
        unit_meta["patient"] = unit_meta[unit_col]
    else:
        unit_meta["patient"] = "NA"

    counts = df.groupby([unit_col, feature_col]).size().reset_index(name="n_cells_feature")
    out = counts.merge(unit_meta, on=unit_col, how="left")
    out = out[out["n_cells_total"] >= min_unit_cells].copy()
    out["proportion"] = out["n_cells_feature"] / out["n_cells_total"]
    out = out.rename(columns={unit_col: "unit_id", feature_col: "feature"})
    out["unit_type"] = unit_col
    out["feature_type"] = feature_col
    return out


def condition_feature_summary(unit_feature_df):
    rows = []
    for (feature, cond), sub in unit_feature_df.groupby(["feature", "condition"]):
        vals = sub["proportion"].astype(float)
        rows.append({
            "feature_type": sub["feature_type"].iloc[0],
            "feature": feature,
            "condition": cond,
            "n_units": int(vals.shape[0]),
            "mean_prop": float(vals.mean()),
            "median_prop": float(vals.median()),
            "sd_prop": float(vals.std(ddof=1)) if vals.shape[0] > 1 else 0.0,
            "min_prop": float(vals.min()),
            "max_prop": float(vals.max()),
            "total_cells_in_units": int(sub["n_cells_total"].sum()),
            "feature_cells_in_units": int(sub["n_cells_feature"].sum()),
        })
    return pd.DataFrame(rows)


def kruskal_by_feature(unit_feature_df):
    rows = []
    conditions = sorted(unit_feature_df["condition"].dropna().unique())
    for feature, sub in unit_feature_df.groupby("feature"):
        groups = [sub.loc[sub["condition"] == cond, "proportion"].values for cond in conditions]
        stat, p, status = safe_kruskal(groups)
        rows.append({
            "feature_type": sub["feature_type"].iloc[0],
            "feature": feature,
            "test": "Kruskal-Wallis",
            "conditions_compared": ";".join(conditions),
            "statistic": stat,
            "pvalue": p,
            "status": status,
            "n_units_total": int(sub["unit_id"].nunique()),
        })
    res = pd.DataFrame(rows)
    if len(res):
        res["qvalue_fdr_bh"] = bh_fdr(res["pvalue"].values)
    return res


def pairwise_mannwhitney_by_feature(unit_feature_df):
    rows = []
    conditions = sorted(unit_feature_df["condition"].dropna().unique())
    for feature, sub in unit_feature_df.groupby("feature"):
        for i, c1 in enumerate(conditions):
            for c2 in conditions[i+1:]:
                x = sub.loc[sub["condition"] == c1, "proportion"].values
                y = sub.loc[sub["condition"] == c2, "proportion"].values
                stat, p, status = safe_mannwhitney(x, y)
                rows.append({
                    "feature_type": sub["feature_type"].iloc[0],
                    "feature": feature,
                    "test": "Mann-Whitney U",
                    "condition_1": c1,
                    "condition_2": c2,
                    "n_units_1": int(len(x)),
                    "n_units_2": int(len(y)),
                    "median_prop_1": float(np.nanmedian(x)) if len(x) else np.nan,
                    "median_prop_2": float(np.nanmedian(y)) if len(y) else np.nan,
                    "mean_prop_1": float(np.nanmean(x)) if len(x) else np.nan,
                    "mean_prop_2": float(np.nanmean(y)) if len(y) else np.nan,
                    "delta_median_1_minus_2": float(np.nanmedian(x) - np.nanmedian(y)) if len(x) and len(y) else np.nan,
                    "statistic": stat,
                    "pvalue": p,
                    "status": status,
                })
    res = pd.DataFrame(rows)
    if len(res):
        res["qvalue_fdr_bh"] = bh_fdr(res["pvalue"].values)
    return res


In [ ]:
# =========================
# Build sample-level tables
# =========================
MIN_SAMPLE_CELLS = 50

sample_comp = build_unit_feature_table(
    obs, sample_col, compartment_col, condition_col,
    dataset_col=dataset_col, patient_col=patient_col,
    min_unit_cells=MIN_SAMPLE_CELLS,
)
sample_ann = build_unit_feature_table(
    obs, sample_col, annotation_col, condition_col,
    dataset_col=dataset_col, patient_col=patient_col,
    min_unit_cells=MIN_SAMPLE_CELLS,
)
sample_cluster = build_unit_feature_table(
    obs, sample_col, cluster_col, condition_col,
    dataset_col=dataset_col, patient_col=patient_col,
    min_unit_cells=MIN_SAMPLE_CELLS,
)

if copykat_col:
    sample_copykat = build_unit_feature_table(
        obs, sample_col, copykat_col, condition_col,
        dataset_col=dataset_col, patient_col=patient_col,
        min_unit_cells=MIN_SAMPLE_CELLS,
    )
else:
    sample_copykat = pd.DataFrame()

SAMPLE_COMP_TSV = METADATA_DIR / f"DA_sample_level_compartment_proportions_{RUN_TAG}.tsv"
SAMPLE_ANN_TSV = METADATA_DIR / f"DA_sample_level_annotation_proportions_{RUN_TAG}.tsv"
SAMPLE_CLUSTER_TSV = METADATA_DIR / f"DA_sample_level_cluster_proportions_{RUN_TAG}.tsv"
SAMPLE_COPYKAT_TSV = METADATA_DIR / f"DA_sample_level_copykat_proportions_{RUN_TAG}.tsv"

write_tsv_replace(sample_comp, SAMPLE_COMP_TSV)
write_tsv_replace(sample_ann, SAMPLE_ANN_TSV)
write_tsv_replace(sample_cluster, SAMPLE_CLUSTER_TSV)
if copykat_col:
    write_tsv_replace(sample_copykat, SAMPLE_COPYKAT_TSV)

display(sample_comp.head())


In [ ]:
# =========================
# Build patient-level tables, if patient_id exists
# =========================
MIN_PATIENT_CELLS = 50

patient_tables = {}
if patient_col:
    patient_tables["compartment"] = build_unit_feature_table(
        obs, patient_col, compartment_col, condition_col,
        dataset_col=dataset_col, patient_col=patient_col,
        min_unit_cells=MIN_PATIENT_CELLS,
    )
    patient_tables["annotation"] = build_unit_feature_table(
        obs, patient_col, annotation_col, condition_col,
        dataset_col=dataset_col, patient_col=patient_col,
        min_unit_cells=MIN_PATIENT_CELLS,
    )
    patient_tables["cluster"] = build_unit_feature_table(
        obs, patient_col, cluster_col, condition_col,
        dataset_col=dataset_col, patient_col=patient_col,
        min_unit_cells=MIN_PATIENT_CELLS,
    )
    if copykat_col:
        patient_tables["copykat"] = build_unit_feature_table(
            obs, patient_col, copykat_col, condition_col,
            dataset_col=dataset_col, patient_col=patient_col,
            min_unit_cells=MIN_PATIENT_CELLS,
        )

    for name, df in patient_tables.items():
        path = METADATA_DIR / f"DA_patient_level_{name}_proportions_{RUN_TAG}.tsv"
        write_tsv_replace(df, path)
else:
    print("[INFO] No patient column found; patient-level tables skipped.")


In [ ]:
# =========================
# Condition summaries
# =========================
summary_comp = condition_feature_summary(sample_comp)
summary_ann = condition_feature_summary(sample_ann)
summary_cluster = condition_feature_summary(sample_cluster)
summary_copykat = condition_feature_summary(sample_copykat) if copykat_col and len(sample_copykat) else pd.DataFrame()

SUMMARY_COMP_TSV = METADATA_DIR / f"DA_condition_summary_compartment_sample_level_{RUN_TAG}.tsv"
SUMMARY_ANN_TSV = METADATA_DIR / f"DA_condition_summary_annotation_sample_level_{RUN_TAG}.tsv"
SUMMARY_CLUSTER_TSV = METADATA_DIR / f"DA_condition_summary_cluster_sample_level_{RUN_TAG}.tsv"
SUMMARY_COPYKAT_TSV = METADATA_DIR / f"DA_condition_summary_copykat_sample_level_{RUN_TAG}.tsv"

write_tsv_replace(summary_comp, SUMMARY_COMP_TSV)
write_tsv_replace(summary_ann, SUMMARY_ANN_TSV)
write_tsv_replace(summary_cluster, SUMMARY_CLUSTER_TSV)
if copykat_col and len(summary_copykat):
    write_tsv_replace(summary_copykat, SUMMARY_COPYKAT_TSV)

display(summary_comp)


In [ ]:
# =========================
# Global Kruskal-Wallis tests
# =========================
kw_comp = kruskal_by_feature(sample_comp)
kw_ann = kruskal_by_feature(sample_ann)
kw_cluster = kruskal_by_feature(sample_cluster)
kw_copykat = kruskal_by_feature(sample_copykat) if copykat_col and len(sample_copykat) else pd.DataFrame()

KW_COMP_TSV = METADATA_DIR / f"DA_global_kruskal_compartment_sample_level_{RUN_TAG}.tsv"
KW_ANN_TSV = METADATA_DIR / f"DA_global_kruskal_annotation_sample_level_{RUN_TAG}.tsv"
KW_CLUSTER_TSV = METADATA_DIR / f"DA_global_kruskal_cluster_sample_level_{RUN_TAG}.tsv"
KW_COPYKAT_TSV = METADATA_DIR / f"DA_global_kruskal_copykat_sample_level_{RUN_TAG}.tsv"

write_tsv_replace(kw_comp.sort_values(["qvalue_fdr_bh", "pvalue"]), KW_COMP_TSV)
write_tsv_replace(kw_ann.sort_values(["qvalue_fdr_bh", "pvalue"]), KW_ANN_TSV)
write_tsv_replace(kw_cluster.sort_values(["qvalue_fdr_bh", "pvalue"]), KW_CLUSTER_TSV)
if copykat_col and len(kw_copykat):
    write_tsv_replace(kw_copykat.sort_values(["qvalue_fdr_bh", "pvalue"]), KW_COPYKAT_TSV)

print("Compartment KW:")
display(kw_comp.sort_values(["qvalue_fdr_bh", "pvalue"]))


In [ ]:
# =========================
# Pairwise Mann-Whitney tests
# =========================
pw_comp = pairwise_mannwhitney_by_feature(sample_comp)
pw_ann = pairwise_mannwhitney_by_feature(sample_ann)
pw_cluster = pairwise_mannwhitney_by_feature(sample_cluster)
pw_copykat = pairwise_mannwhitney_by_feature(sample_copykat) if copykat_col and len(sample_copykat) else pd.DataFrame()

PW_COMP_TSV = METADATA_DIR / f"DA_pairwise_mannwhitney_compartment_sample_level_{RUN_TAG}.tsv"
PW_ANN_TSV = METADATA_DIR / f"DA_pairwise_mannwhitney_annotation_sample_level_{RUN_TAG}.tsv"
PW_CLUSTER_TSV = METADATA_DIR / f"DA_pairwise_mannwhitney_cluster_sample_level_{RUN_TAG}.tsv"
PW_COPYKAT_TSV = METADATA_DIR / f"DA_pairwise_mannwhitney_copykat_sample_level_{RUN_TAG}.tsv"

write_tsv_replace(pw_comp.sort_values(["qvalue_fdr_bh", "pvalue"]), PW_COMP_TSV)
write_tsv_replace(pw_ann.sort_values(["qvalue_fdr_bh", "pvalue"]), PW_ANN_TSV)
write_tsv_replace(pw_cluster.sort_values(["qvalue_fdr_bh", "pvalue"]), PW_CLUSTER_TSV)
if copykat_col and len(pw_copykat):
    write_tsv_replace(pw_copykat.sort_values(["qvalue_fdr_bh", "pvalue"]), PW_COPYKAT_TSV)

print("Top pairwise compartment differences:")
display(pw_comp.sort_values(["qvalue_fdr_bh", "pvalue"]).head(20))


In [ ]:
# =========================
# Secondary binomial GLM sensitivity analysis
# =========================
# This is not the primary test; it is a sensitivity analysis.
# It models feature_cells / total_cells per sample as a function of condition.

def glm_binomial_condition(unit_feature_df):
    if not HAS_STATSMODELS:
        return pd.DataFrame([{"status": "statsmodels_not_available"}])

    rows = []
    for feature, sub in unit_feature_df.groupby("feature"):
        sub = sub.copy()
        if sub["condition"].nunique() < 2 or sub.shape[0] < 4:
            rows.append({"feature": feature, "status": "insufficient_conditions_or_units"})
            continue
        try:
            y = sub["proportion"].astype(float).values
            X = pd.get_dummies(sub["condition"].astype(str), prefix="condition", drop_first=True)
            X = sm.add_constant(X, has_constant="add")
            weights = sub["n_cells_total"].astype(float).values

            model = sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=weights)
            fit = model.fit(maxiter=100)

            null_X = np.ones((len(y), 1))
            null = sm.GLM(y, null_X, family=sm.families.Binomial(), freq_weights=weights).fit(maxiter=100)
            lr_stat = 2 * (fit.llf - null.llf)
            df_diff = fit.df_model - null.df_model
            p = stats.chi2.sf(lr_stat, df_diff) if df_diff > 0 else np.nan

            rows.append({
                "feature_type": sub["feature_type"].iloc[0],
                "feature": feature,
                "test": "binomial_GLM_condition_LRT",
                "n_units": int(sub.shape[0]),
                "df_model": float(fit.df_model),
                "lr_statistic": float(lr_stat),
                "pvalue": float(p),
                "status": "ok",
            })
        except Exception as e:
            rows.append({"feature": feature, "test": "binomial_GLM_condition_LRT", "pvalue": np.nan, "status": f"error:{repr(e)}"})

    res = pd.DataFrame(rows)
    if "pvalue" in res.columns:
        res["qvalue_fdr_bh"] = bh_fdr(res["pvalue"].values)
    return res

glm_comp = glm_binomial_condition(sample_comp)
glm_ann = glm_binomial_condition(sample_ann)
glm_cluster = glm_binomial_condition(sample_cluster)

GLM_COMP_TSV = METADATA_DIR / f"DA_glm_binomial_compartment_sample_level_{RUN_TAG}.tsv"
GLM_ANN_TSV = METADATA_DIR / f"DA_glm_binomial_annotation_sample_level_{RUN_TAG}.tsv"
GLM_CLUSTER_TSV = METADATA_DIR / f"DA_glm_binomial_cluster_sample_level_{RUN_TAG}.tsv"

write_tsv_replace(glm_comp.sort_values(["qvalue_fdr_bh", "pvalue"], na_position="last") if "qvalue_fdr_bh" in glm_comp.columns else glm_comp, GLM_COMP_TSV)
write_tsv_replace(glm_ann.sort_values(["qvalue_fdr_bh", "pvalue"], na_position="last") if "qvalue_fdr_bh" in glm_ann.columns else glm_ann, GLM_ANN_TSV)
write_tsv_replace(glm_cluster.sort_values(["qvalue_fdr_bh", "pvalue"], na_position="last") if "qvalue_fdr_bh" in glm_cluster.columns else glm_cluster, GLM_CLUSTER_TSV)

display(glm_comp.sort_values(["qvalue_fdr_bh", "pvalue"], na_position="last") if "qvalue_fdr_bh" in glm_comp.columns else glm_comp)


In [ ]:
# =========================
# Per-cell contingency tables for descriptive support only
# =========================
# These are not the main condition-level tests, but useful QC summaries.

cont_rows = []
for feature_col, label in [(compartment_col, "compartment"), (annotation_col, "annotation"), (cluster_col, "cluster")]:
    tab = pd.crosstab(obs[feature_col].astype(str), obs[condition_col].astype(str))
    chi2, p, dof, exp = stats.chi2_contingency(tab.values)
    cv = cramers_v_from_table(tab.values)
    cont_rows.append({
        "feature_type": label,
        "table": f"{feature_col} x {condition_col}",
        "test": "chi_square_independence_per_cell_descriptive",
        "n_cells": int(tab.values.sum()),
        "n_rows": int(tab.shape[0]),
        "n_cols": int(tab.shape[1]),
        "chi2": float(chi2),
        "dof": int(dof),
        "pvalue": float(p),
        "cramers_v": float(cv),
        "interpretation_note": "Descriptive per-cell association only; sample-level tests are primary for condition comparisons.",
    })

cont_df = pd.DataFrame(cont_rows)
CONT_TSV = METADATA_DIR / f"DA_per_cell_condition_association_descriptive_{RUN_TAG}.tsv"
write_tsv_replace(cont_df, CONT_TSV)
display(cont_df)


In [ ]:
# =========================
# Figures: condition composition stacked bars
# =========================
figure_manifest = []

def plot_condition_stacked(obs, feature_col, title, filename):
    tab = pd.crosstab(obs[condition_col].astype(str), obs[feature_col].astype(str), normalize="index")
    ax = tab.plot(kind="bar", stacked=True, figsize=(10, 5), width=0.85)
    ax.set_ylabel("Fraction of cells")
    ax.set_xlabel("Condition")
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    path = FIGURES_DIR / filename
    savefig_replace(path)
    figure_manifest.append({"figure": str(path), "type": "stacked_bar", "feature_col": feature_col})

plot_condition_stacked(obs, compartment_col, "Cell fraction by final compartment and condition", f"DA_condition_stacked_final_compartment_{RUN_TAG}.png")
plot_condition_stacked(obs, annotation_col, "Cell fraction by refined annotation and condition", f"DA_condition_stacked_refined_annotation_{RUN_TAG}.png")
plot_condition_stacked(obs, cluster_col, "Cell fraction by cluster and condition", f"DA_condition_stacked_cluster_{RUN_TAG}.png")
if copykat_col:
    plot_condition_stacked(obs, copykat_col, "Cell fraction by CopyKAT call and condition", f"DA_condition_stacked_copykat_{RUN_TAG}.png")


In [ ]:
# =========================
# Figures: sample-level boxplots for final compartments
# =========================

def plot_sample_box(unit_df, title, filename, max_features=12):
    feat_order = (
        unit_df.groupby("feature")["proportion"].mean()
        .sort_values(ascending=False)
        .head(max_features)
        .index.tolist()
    )
    sub = unit_df[unit_df["feature"].isin(feat_order)].copy()
    conditions = sorted(sub["condition"].unique())

    n_feat = len(feat_order)
    fig, axes = plt.subplots(n_feat, 1, figsize=(10, max(3, 2.2 * n_feat)), sharex=True)
    if n_feat == 1:
        axes = [axes]

    for ax, feat in zip(axes, feat_order):
        data = [sub.loc[(sub["feature"] == feat) & (sub["condition"] == cond), "proportion"].values for cond in conditions]
        ax.boxplot(data, labels=conditions, showfliers=False)
        for i, vals in enumerate(data, start=1):
            if len(vals):
                jitter = np.random.default_rng(42).normal(0, 0.04, size=len(vals))
                ax.scatter(np.full(len(vals), i) + jitter, vals, s=12, alpha=0.65)
        ax.set_ylabel(feat[:35])
        ax.set_ylim(bottom=0)
    axes[0].set_title(title)
    axes[-1].set_xlabel("Condition")
    path = FIGURES_DIR / filename
    savefig_replace(path)
    figure_manifest.append({"figure": str(path), "type": "sample_level_boxplot", "feature_col": unit_df["feature_type"].iloc[0]})

plot_sample_box(sample_comp, "Sample-level final compartment proportions by condition", f"DA_sample_box_final_compartment_{RUN_TAG}.png", max_features=8)
plot_sample_box(sample_cluster, "Sample-level cluster proportions by condition", f"DA_sample_box_clusters_{RUN_TAG}.png", max_features=12)


In [ ]:
# =========================
# Figures: sample-level heatmap of final compartments
# =========================

def plot_sample_heatmap(unit_df, title, filename):
    pivot = unit_df.pivot_table(index="unit_id", columns="feature", values="proportion", fill_value=0)
    meta = unit_df[["unit_id", "condition"]].drop_duplicates().set_index("unit_id")
    if "malignant_like" in pivot.columns:
        sort_key = pivot["malignant_like"]
    else:
        sort_key = pivot.sum(axis=1)
    order = meta.assign(sort_key=sort_key).sort_values(["condition", "sort_key"], ascending=[True, False]).index
    pivot = pivot.loc[order]

    fig, ax = plt.subplots(figsize=(max(8, 0.14 * pivot.shape[0]), max(4, 0.35 * pivot.shape[1])))
    im = ax.imshow(pivot.T.values, aspect="auto", interpolation="nearest")
    ax.set_yticks(range(pivot.shape[1]))
    ax.set_yticklabels(pivot.columns, fontsize=8)
    ax.set_xticks(range(pivot.shape[0]))
    ax.set_xticklabels(pivot.index, rotation=90, fontsize=6)
    ax.set_title(title)
    ax.set_xlabel("Sample")
    ax.set_ylabel("Feature")
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Proportion")
    path = FIGURES_DIR / filename
    savefig_replace(path)
    figure_manifest.append({"figure": str(path), "type": "sample_level_heatmap", "feature_col": unit_df["feature_type"].iloc[0]})

plot_sample_heatmap(sample_comp, "Sample-level composition heatmap: final compartments", f"DA_sample_heatmap_final_compartment_{RUN_TAG}.png")


In [ ]:
# =========================
# Save figure manifest and output manifest
# =========================
FIGURE_MANIFEST_TSV = METADATA_DIR / f"DA_figure_manifest_{RUN_TAG}.tsv"
write_tsv_replace(pd.DataFrame(figure_manifest), FIGURE_MANIFEST_TSV)

outputs = [
    READINESS_TSV,
    SAMPLE_COMP_TSV,
    SAMPLE_ANN_TSV,
    SAMPLE_CLUSTER_TSV,
    SUMMARY_COMP_TSV,
    SUMMARY_ANN_TSV,
    SUMMARY_CLUSTER_TSV,
    KW_COMP_TSV,
    KW_ANN_TSV,
    KW_CLUSTER_TSV,
    PW_COMP_TSV,
    PW_ANN_TSV,
    PW_CLUSTER_TSV,
    GLM_COMP_TSV,
    GLM_ANN_TSV,
    GLM_CLUSTER_TSV,
    CONT_TSV,
    FIGURE_MANIFEST_TSV,
]
if copykat_col:
    outputs.extend([SAMPLE_COPYKAT_TSV, SUMMARY_COPYKAT_TSV, KW_COPYKAT_TSV, PW_COPYKAT_TSV])

if patient_col:
    outputs.extend(list(METADATA_DIR.glob(f"DA_patient_level_*_proportions_{RUN_TAG}.tsv")))

manifest = pd.DataFrame({
    "path": [str(p) for p in outputs],
    "exists": [Path(p).exists() for p in outputs],
    "size_bytes": [Path(p).stat().st_size if Path(p).exists() else np.nan for p in outputs],
})

OUTPUT_MANIFEST_TSV = METADATA_DIR / f"DA_composition_output_manifest_{RUN_TAG}.tsv"
write_tsv_replace(manifest, OUTPUT_MANIFEST_TSV)
display(manifest)


## Interpretation notes

- Kruskal-Wallis tests are used as the primary global test because sample-level proportions are often non-normal and sample counts are limited.
- Pairwise Mann-Whitney U tests identify which condition pairs differ after the global test.
- FDR correction is used because multiple compartments, annotations, and clusters are tested.
- Per-cell chi-square tests are included only as descriptive support, because cells from the same sample/patient are not fully independent.
- The binomial GLM is included as a secondary sensitivity analysis and should not replace sample-level nonparametric testing.
